# Aula 6 — Quartis, outliers, normalização e padronização

**Habilidade:** Calcular e interpretar medidas descritivas e de variação, explicando o que significam no contexto do problema.

**Objetos:** Quartis e outliers. Normalização e padronização (conceito e quando usar).

Aplique a regra do IQR à temperatura (ano todo) e à cota (por período), teste o salto diário e reescale a cota.

> Base **simulada** (didática). Se o download falhar, envie `rio_negro_cotas.csv` pelo painel de arquivos do Colab.

In [ ]:
# Passo 0 — baixar o CSV do projeto (executar uma vez)
import os, urllib.request
if not os.path.exists("rio_negro_cotas.csv"):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/priscilagon/matematica-estatistica-ia/main/dados/rio_negro_cotas.csv", "rio_negro_cotas.csv")
print("CSV pronto:", os.path.exists("rio_negro_cotas.csv"))

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("rio_negro_cotas.csv", parse_dates=["data"])

# Aula 6 — Quartis, outliers (regra do IQR), normalização e padronização
def limites_iqr(s):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    return q1, q3, iqr, q1 - 1.5 * iqr, q3 + 1.5 * iqr

# 1) Temperatura da água: análise do ano todo
t = df["temperatura_c"]
q1, q3, iqr, li, ls = limites_iqr(t)
print("Temperatura  Q1=%.2f Q3=%.2f IQR=%.2f  limites=[%.2f ; %.2f]" % (q1, q3, iqr, li, ls))
print(df.loc[(t < li) | (t > ls), ["data", "temperatura_c"]])

# 2) Cota: o contexto importa — a regra é aplicada DENTRO de cada período
print()
for p, g in df.groupby("periodo"):
    q1, q3, iqr, li, ls = limites_iqr(g["cota_metros"])
    atip = g[(g["cota_metros"] < li) | (g["cota_metros"] > ls)]
    print("%-9s limites=[%.2f ; %.2f]  atípicos: %s" %
          (p, li, ls, list(zip(atip["data"].dt.strftime("%d/%m"), atip["cota_metros"]))))

# 3) Atípico estatístico NÃO é o mesmo que ERRO. Teste do salto diário:
#    um rio não sobe nem desce 2 m de um dia para o outro.
salto = df["cota_metros"].diff().abs()
print("\nSaltos maiores que 2 m entre dias seguidos:")
print(df.loc[salto > 2, ["data", "cota_metros"]])

# 4) Normalização (min-máx) e padronização (z-score) da cota
x = df["cota_metros"]
x_mm = (x - x.min()) / (x.max() - x.min())
z = (x - x.mean()) / x.std(ddof=0)       # mesmo critério do StandardScaler
print("\nmin-máx: mínimo=%.2f máximo=%.2f" % (x_mm.min(), x_mm.max()))
print("z-score: média=%.2f desvio=%.2f" % (round(z.mean(), 6) + 0.0, z.std(ddof=0)))

# 5) O outlier "achata" a escala min-máx: compare com e sem as leituras atípicas
ok = x[(x > 10) & (x < 33)]
print("\nCom atípicos: 20,00 m vira %.3f" % ((20 - x.min()) / (x.max() - x.min())))
print("Sem atípicos: 20,00 m vira %.3f" % ((20 - ok.min()) / (ok.max() - ok.min())))

## O que observar

- A temperatura de **38,5 °C (22/10)** é o único atípico do ano; limites 25,55 e 31,55 °C.
- A regra por período acusa 18/03 (enchente) e 09/08 (vazante) e também **quatro dias de início de outubro** na seca — um alerta sem erro (teste do salto diário os absolve).
- `min-máx` termina em 0 e 1; o `z-score` tem média 0 e DP 1.
- A comparação “com e sem atípicos” mostra o valor 20 m indo de 0,424 para 0,322.

## Atividades de fixação

1. Para 12, 14, 15, 16, 17, 18, 19, 21, 45, calcule Q1, Q3, IQR e os limites de Tukey. Existe outlier?
2. Normalize pelo método min-máx os valores 10, 20, 30, 50.
3. Padronize os valores 10, 20, 30, 50 (use DP amostral).
4. Aplique a regra do IQR à temperatura da água. Quais são os limites e o que você faria com o valor atípico?
5. O dia 18/03 tem cota de 8,90 m, e o dia seguinte, 23,80 m. É um outlier “real”? Justifique.
6. Por que a normalização min-máx é mais sensível a outliers do que a mediana e o IQR?


## Minhas respostas

(Use células de texto e de código abaixo.)